# 🚀  How to Build & Test Your Streamlit App in Colab
This notebook is a template for developing and testing your Streamlit app. You will use this to write your code, test it live, and then download the final files to upload to GitHub.



<img src="https://raw.githubusercontent.com/sasrwt/homequity/main/streamlitappflow.png" width="500">

### **Step 1: Install All Libraries**
Run the code cell below. This installs Streamlit, the libraries your app needs (pandas, scikit-learn), and localtunnel (creates a temporary public URL that "tunnels" to the hmeq app running insidColab notebook. This makes our app accessible in any web browser.).

In [1]:
# Install all required libraries
!pip install streamlit pandas scikit-learn -q
!npm install localtunnel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 78.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

### **Step 2: Write Your App Code (`app.py`)**

https://docs.streamlit.io/develop/api-reference/widgets

Run the cell below.  
The `%%writefile` command will take all the code in the cell and save it as a new file named **`app.py`**.

This is where you will do your main work. You need to:

← **Update the `st.slider` and `st.selectbox` widgets** to match the input columns your model requires.

← **Update the `pd.DataFrame`** so its columns match the widgets you created.

← **Change the file path** so it correctly points to the location of your uploaded **`.pkl`** model file.


In [ ]:
%%writefile hmeqapp.py

# -*- coding: utf-8 -*-
import streamlit as st
import pickle
import pandas as pd
import sklearn  # This is needed for the pickle file to load!

# Load the trained model
# --- Put the Model in Drive First---
with open("/content/my_model.pkl", "rb") as file:
    model = pickle.load(file)

# Title for the app
st.markdown(
    "<h1 style='text-align: center; background-color: #f0f8ff; padding: 20px; color: #4682b4; border-radius: 15px; font-family: \"'Roboto'\", sans-serif; box-shadow: 3px 3px 10px rgba(0,0,0,0.2);'>🏦💰 Loan Approval Predictor 📈📊</h1>",
    unsafe_allow_html=True
)

# Numeric inputs
st.header("Enter Loan Applicant's Details")

# Input fields for numerical values based on the model's features
fico_score = st.number_input("FICO Score", min_value=300, max_value=850, value=700, step=1, help="Enter the applicant's credit score (300-850).")
granted_loan_amount = st.number_input("Granted Loan Amount", min_value=5000, max_value=100000, value=25000, step=1000, help="The amount of loan granted.")
monthly_gross_income = st.number_input("Monthly Gross Income", min_value=500, max_value=20000, value=5000, step=100, help="Applicant's total monthly income before deductions.")
monthly_housing_payment = st.number_input("Monthly Housing Payment", min_value=100, max_value=10000, value=1500, step=50, help="Applicant's total monthly housing expenses.")
ever_bankrupt_or_foreclose = st.radio("Ever Bankrupt or Foreclose", options=[0, 1], format_func=lambda x: "No" if x == 0 else "Yes", horizontal=True, help="Has the applicant ever been bankrupt or foreclosed?")

# Categorical inputs with options based on the model's features
reason = st.selectbox("Reason for Loan", [
    'credit_card_refinancing', 'debt_conslidation', 'home_improvement',
    'major_purchase', 'cover_an_unexpected_cost', 'other'
], help="The primary reason for requesting the loan.")
employment_status = st.selectbox("Employment Status", ['full_time', 'unemployed', 'part_time', 'self_employed'], help="Applicant's current employment status.")
employment_sector = st.selectbox("Employment Sector", [
    'industrials', 'other', 'real_estate', 'financials', 'consumer_staples',
    'consumer_discretionary', 'telecommunication_services', 'materials',
    'health_care', 'utilities', 'information_technology', 'Unknown', 'energy'
], help="The industry sector of the applicant's employer.")
lender = st.selectbox("Lender", ['A', 'B', 'C'], help="The specific lender offering the loan.")

# Create the input data as a DataFrame
input_data = pd.DataFrame({
    "FICO_score": [fico_score],
    "Granted_Loan_Amount": [granted_loan_amount],
    "Monthly_Gross_Income": [monthly_gross_income],
    "Monthly_Housing_Payment": [monthly_housing_payment],
    "Ever_Bankrupt_or_Foreclose": [ever_bankrupt_or_foreclose],
    "Reason": [reason],
    "Employment_Status": [employment_status],
    "Employment_Sector": [employment_sector],
    "Lender": [lender]
})

# --- Prepare Data for Prediction ---
# 1. One-hot encode the user's input.
categorical_cols = ['Reason', 'Employment_Status', 'Employment_Sector', 'Lender']
input_data_encoded = pd.get_dummies(input_data, columns=categorical_cols, drop_first=True)

# 2. Add any "missing" columns the model expects (fill with 0).
model_columns = model.feature_names_in_
for col in model_columns:
    if col not in input_data_encoded.columns:
        input_data_encoded[col] = 0

# 3. Reorder/filter columns to exactly match the model's training data.
input_data_encoded = input_data_encoded[model_columns]

# Predict button
if st.button("Evaluate Loan"):
    # Predict using the loaded model
    prediction = model.predict(input_data_encoded)[0]
    prediction_proba = model.predict_proba(input_data_encoded)[:, 1][0]

    # Display result
    if prediction == 1:
        st.success(f"✅ The model predicts: **Approved!** (Probability: {prediction_proba:.2f})")
    else:
        st.error(f"❌ The model predicts: **Denied.** (Probability: {prediction_proba:.2f})")


Overwriting hmeqapp.py


### **Step 3: Write Your Requirements File**

Need this file in GitHub.  
It tells **Streamlit Cloud** which libraries need to be installed.

Run the cell below to create **`requirements.txt`**.

← **Important:** If your model uses additional libraries (such as `xgboost`, `lightgbm`, `catboost`, etc.),  
you **must manually add them** to the list before deploying!


In [ ]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn
numpy
matplotlib
seaborn
scipy

Writing requirements.txt


### **Step 4: Test Your App Live!**

Run the code cell below.  
It will start your **Streamlit app** in the background and then give you a **public URL** to test it.

← Click the URL (it will look something like `https://some-random-words.loca.lt`).

← When the new page opens, click the **"Click to Continue"** button.

← Your app will appear! **Test it thoroughly** to make sure all inputs, predictions, and outputs work as expected.


In [ ]:
# Run the app in the background, redirecting output to logs.txt
!streamlit run hmeqapp.py &>/content/logs.txt &

# Give Streamlit a few seconds to start up
!sleep 5

# Get the password (your IP) and print it.
# The "&" runs it in the background while the tunnel starts.
print("Fetching your tunnel password...")
!curl https://loca.lt/mytunnelpassword &

# Start the tunnel. This command will keep the cell running and give you the URL.
!npx localtunnel --port 8501

Fetching your tunnel password...
34.75.223.30⠙your url is: https://twenty-moons-shine.loca.lt
^C


## Kick the Can and Move On


That TypeError: Failed to fetch is a browser-side error. It means your browser (the app's frontend) tried to download its own JavaScript files from the localtunnel URL, but the tunnel was so slow or unstable that the request failed.

This confirms our theory: your app is taking too long to load the model, and the localtunnel connection is collapsing or timing out before the app can become stable.

💡 The Real Problem
The "Colab + localtunnel" workflow is just a testing tool. It is famously unstable and difficult, as you've seen.

We have been using this flaky, complicated method (managing passwords, running multiple cells, curl commands) just to test the app. But this test is now causing more problems than it's worth.

The good news is we don't need it anymore. We have already learned everything we need from this Colab experiment:

We have the final, working hmeqapp.py code.

We have the final, working requirements.txt file (after fixing the sklearn typo).

We know the hmeq_model.pkl file works.

The goal was never to use the Colab app. The goal was to get the files ready for a real deployment. That mission is accomplished.

### **Step 6: Download Your Files for GitHub**

If your app works, you're ready for deployment.

← Go to the **Files** sidebar on the left.

← Find **`app.py`**. Right-click it and select **Download**.

← Find **`requirements.txt`**. Right-click it and select **Download**.

You are now ready to upload **`app.py`**, **`requirements.txt`**, and your **`.pkl` model file** to a new GitHub repository for deployment.

---

### **Final GitHub Step (Important!)**

Before deploying, edit your **`app.py`** file **one last time** (either in GitHub or in a local editor):

← **Remove the folder path** from your `open()` command.  
It should look like this:

```python
open("your_model_name.pkl", "rb")
